# Trajectory Visualization -- Module 1 + 2 + 3 Combined Output

Saves labeled panel figures of Module 2's synthesized multi-stage trajectory, with Module 3's
per-step "probability of further progression within 2 years" captioned beneath each panel.

`train_module2_poc.py` only saves a single-step training-sanity strip, and
`evaluate_trajectory.py` reduces the synthesized images straight to PSNR/SSIM/FID numbers
without saving them. This notebook is what keeps the actual images (and Module 3's estimate on
each one).

**Run this after notebooks 05, 06, AND 07 have all finished** -- it reads the Module 1 cache,
the trained Module 2 Generator checkpoint, and the trained Module 3 checkpoint + its saved LBS
thresholds, all from Drive/the current runtime's local outputs.

Module 3 estimates a probability within a fixed 2-year horizon, never an exact date. Every
synthesized (non-baseline) panel's Module 3 estimate is captioned "(synthesized image --
untested)" -- Module 3 is trained only on real photographs, so running it on Module 2's
synthesized output is mechanically possible but was not part of its training distribution.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'main'  # <-- change if this work isn't merged to main yet
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
TIANJIN_CACHE = os.path.join(DRIVE_DATA_DIR, 'module1_cache', 'module1_outputs_tianjin.pt')
assert os.path.isfile(TIANJIN_CACHE), f"Not found: {TIANJIN_CACHE} -- run notebook 05 first.

In [ ]:
# Unzip Tianjin locally (same convention as notebooks 05/06/07).
import glob

os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/retinal-dr-longitudinal.zip" -d _tianjin_extract_raw

_manifest_candidates = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _manifest_candidates, 'No corrected_manifest.csv found -- see notebook 05 for troubleshooting.'
_tianjin_root = os.path.dirname(_manifest_candidates[0])
TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if _tianjin_root != TIANJIN_DIR and not os.path.exists(TIANJIN_DIR):
    os.symlink(_tianjin_root, TIANJIN_DIR)

# Same laterality-resolution fetch-or-compute as notebooks 05/06/07 -- required by
# tianjin_dataset.py / module3/dataset.py.
LATERALITY_DRIVE_PATH = os.path.join(DRIVE_DATA_DIR, 'module1_cache', 'laterality_resolved.csv')
LATERALITY_LOCAL_PATH = os.path.join(TIANJIN_DIR, 'laterality_resolved.csv')
assert os.path.isfile(LATERALITY_DRIVE_PATH), (
    f"Not found: {LATERALITY_DRIVE_PATH} -- run notebook 05 first (it computes and persists this)."
)
import shutil
shutil.copy(LATERALITY_DRIVE_PATH, LATERALITY_LOCAL_PATH)
print('Tianjin manifest found:', os.path.isfile(os.path.join(TIANJIN_DIR, 'corrected_manifest.csv')))

In [ ]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}

REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"

%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch matplotlib

In [ ]:
# Locate the Module 1 classifier + EX/MA (+ HE/SE if trained) checkpoints, same discovery
# logic as notebooks 04/05/06.
import glob

SEG_DIR   = '/content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation'
CLS_SAVES = '/content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/fgadr/saves'

_pref  = ['final_weights.pt', 'best_validation_weights.pt']
_cands = [os.path.join(CLS_SAVES, n) for n in _pref] + sorted(
    glob.glob(os.path.join(CLS_SAVES, '*.pt')), key=os.path.getmtime, reverse=True)
CLS_CKPT = next((p for p in _cands if os.path.isfile(p)), None)
assert CLS_CKPT, f"No classifier .pt in {CLS_SAVES} -- run notebook 02 first"

EX_CKPT = os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_ex', 'model_2.pth.tar')
MA_CKPT = os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_ma', 'model_2.pth.tar')
for _p in (CLS_CKPT, EX_CKPT, MA_CKPT):
    assert os.path.isfile(_p), f"missing checkpoint: {_p} -- run notebooks 02/03 first"

HE_CKPT = os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_he', 'model_2.pth.tar')
SE_CKPT = os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_se', 'model_2.pth.tar')
_extra_seg_args = ''
if os.path.isfile(HE_CKPT):
    _extra_seg_args += f' --seg-checkpoint HE={HE_CKPT}'
if os.path.isfile(SE_CKPT):
    _extra_seg_args += f' --seg-checkpoint SE={SE_CKPT}'
print('classifier :', CLS_CKPT)
print('EX seg     :', EX_CKPT)
print('MA seg     :', MA_CKPT)
print('HE seg     :', HE_CKPT if os.path.isfile(HE_CKPT) else 'not trained yet')
print('SE seg     :', SE_CKPT if os.path.isfile(SE_CKPT) else 'not trained yet')

# Module 2 + Module 3 checkpoints, from notebooks 06 and 07's local runtime outputs (or copy
# them here from Drive first if this is a fresh runtime from those notebooks).
GENERATOR_CKPT = f'{REPO_CODE_DIR}/DRForestGAN-v2/stargan/models_poc/final-G.ckpt'
MODULE3_CKPT = f'{REPO_CODE_DIR}/module3/module3_runs_poc/final-model.ckpt'
MODULE3_RESULTS_JSON = f'{REPO_CODE_DIR}/module3/module3_runs_poc/poc_results.json'
for _p in (GENERATOR_CKPT, MODULE3_CKPT, MODULE3_RESULTS_JSON):
    assert os.path.isfile(_p), (
        f"Not found: {_p} -- run notebook 06 (Module 2) and notebook 07 (Module 3) in this "
        "same runtime first, or copy their checkpoints here."
    )

In [ ]:
%cd {REPO_CODE_DIR}
!python visualize_trajectory.py \
    --tianjin-dir "{TIANJIN_DIR}" \
    --tianjin-module1-cache "{TIANJIN_CACHE}" \
    --generator-checkpoint "{GENERATOR_CKPT}" \
    --classifier-checkpoint "{CLS_CKPT}" \
    --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
    --module3-checkpoint "{MODULE3_CKPT}" \
    --module3-thresholds-json "{MODULE3_RESULTS_JSON}" \
    --num-patients 4 \
    --out-dir ./trajectory_figures/

In [ ]:
from IPython.display import Image as IPImage, display
import glob

figures = sorted(glob.glob(f'{REPO_CODE_DIR}/trajectory_figures/patient_*.png'))
print(f'{len(figures)} figure(s) saved:')
for fig_path in figures:
    print(' ', fig_path)
    display(IPImage(fig_path))

## Caveats

- These figures show real progression pairs from Tianjin only (patients whose real follow-up
  grade is higher than their real baseline grade) -- the cascade is only defined for
  progression, not regression or no-change.
- Module 3's caption is "estimated probability of further progression within 2 years," never
  a specific date -- that is the only horizon the model has real training signal for.
- Every synthesized (non-baseline) panel's Module 3 estimate is marked "(synthesized image --
  untested)": Module 3 is trained only on real baseline photographs, and running it on a
  Module-2-synthesized image is mechanically possible but was not part of its training
  distribution.
- Module 1's "consistency" flag on each synthesized panel means the re-graded synthesized
  image matches the intended target stage -- a False here could mean the generator produced
  something implausible, or that Module 1's own grader is unreliable on synthesized
  (out-of-distribution) images, or both.